<a href="https://colab.research.google.com/github/Mouad-Tazka/-ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-25%20%E2%80%94%20Cleaning%20Gauntlet%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
before = len(df)

df = df.drop_duplicates().copy()

rows_removed = before - len(df)
log(
    'duplicates',
    'Removed exact duplicate rows',
    rows_removed
)

[duplicates] Removed exact duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
rows_affected = len(df)

df['price'] = pd.to_numeric(
    df['price'].str.replace('$', '', regex=False),
    errors='raise'
).astype(float)

log(
    'price',
    'Removed dollar signs and converted price to float',
    rows_affected
)

[price] Removed dollar signs and converted price to float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')

invalid_qty = df['qty'].isna() | (df['qty'] < 1)
rows_removed = int(invalid_qty.sum())

df = df.loc[~invalid_qty].copy()

log(
    'qty',
    'Dropped rows with missing or non-positive quantity',
    rows_removed
)

[qty] Dropped rows with missing or non-positive quantity (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
print(df['item'].value_counts())

ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho'
}

before_item = df['item'].copy()

df['item'] = df['item'].map(ITEM_MAP)

rows_changed = int((before_item != df['item']).sum())

log(
    'item',
    'Collapsed six item spellings into three canonical product names',
    rows_changed
)

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] Collapsed six item spellings into three canonical product names (126 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
import unicodedata # Import unicodedata for string normalization

print(df['category'].value_counts())

CATEGORY_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'Merch': 'Merch',
    'Apparel': 'Apparel',
    'RainGear': 'Rain Gear',
    'rain-gear': 'Rain Gear'
}

before_category = df['category'].copy()

df['category'] = df['category'].astype(str).str.strip().apply(lambda x: unicodedata.normalize('NFKC', x)).map(CATEGORY_MAP)

rows_changed = int((before_category != df['category']).sum())

log(
    'category_format',
    'Normalized capitalization and formatting of category names',
    rows_changed
)

# Business decision: combine Apparel with Merch
apparel_rows = int((df['category'] == 'Apparel').sum())

df.loc[df['category'] == 'Apparel', 'category'] = 'Merch'

log(
    'category_business',
    'Grouped Apparel into the broader Merch category for reporting',
    apparel_rows
)

print(df['category'].value_counts())
print(df['category'].unique())

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category_format] Normalized capitalization and formatting of category names (130 row(s))
[category_business] Grouped Apparel into the broader Merch category for reporting (43 row(s))
category
Food         95
Merch        94
Rain Gear    86
Name: count, dtype: int64
['Rain Gear' 'Merch' 'Food']


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float

assert set(df['item'].unique()) == {
    'Cheeseburger',
    'Foam Finger',
    'Rain Poncho'
}

assert set(df['category'].unique()) == {
    'Food',
    'Merch',
    'Rain Gear'
}

assert df['order_id'].is_unique

print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [11]:
df['revenue'] = df['qty'] * df['price']

revenue_by_category = (
    df.groupby('category')['revenue']
      .sum()
      .sort_values(ascending=False)
      .round(2)
)

print('Revenue by category:')
for category, revenue in revenue_by_category.items():
    print(f'{category}: ${revenue:,.2f}')

total_revenue = round(df['revenue'].sum(), 2)

print(f'\nOverall total: ${total_revenue:,.2f}')

Revenue by category:
Food: $1,656.00
Merch: $1,572.00
Rain Gear: $1,512.00

Overall total: $4,740.00


**What I would tell the vendor:** Based on revenue alone, I would tell the vendor to stock more Food-category items because Food generated the most revenue at $1,656.00.

### TODO 8 — read back your log

In [12]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,Removed exact duplicate rows,15
1,price,Removed dollar signs and converted price to float,300
2,qty,Dropped rows with missing or non-positive quan...,25
3,item,Collapsed six item spellings into three canoni...,126
4,category_format,Normalized capitalization and formatting of ca...,130
5,category_business,Grouped Apparel into the broader Merch categor...,43


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a) Dropping the duplicate rows changed the revenue total the most. Before removing duplicates, revenue was \$4,852.50, and afterward it was \$4,594.50, a decrease of \$258.00.

b) One decision that could reasonably have gone another way was grouping Apparel into Merch. I chose to combine them because I wanted the vendor report to use broader merchandise categories. If I had kept Apparel separate, the overall revenue would still have been \$4,740.00, but the category report would have shown \$856.50 in Merch and \$715.50 in Apparel instead of \$1,572.00 in the combined Merch category. Keeping them separate would also have been reasonable, but combining them made more sense for the broader category-level report I was trying to produce.
